# Group 22 Google Play Store exploration

Collaborative CA6002 assignment. This supplied notebook covers exploration and preprocessing only. It does not reproduce the v5 regression benchmarks or final model. Outputs have been cleared; initialization order and the dataset path were corrected during repository preparation. See ../docs/REVIEW_NOTES.md for version differences. Run from the repository root or the notebooks directory.


## Source acknowledgement

This Group 22 assignment draws on and adapts content from data13's Kaggle notebook [Machine Learning Model to Predict App Rating (94%)](https://www.kaggle.com/code/data13/machine-learning-model-to-predict-app-rating-94). We acknowledge this reference for the assignment's analysis and modelling approach. Credit for borrowed content belongs to the original author; the assignment should not be presented as wholly original work by Group 22. The exact extent of reuse has not been mapped to individual code cells, text or figures. The 94% in the source title is not a performance claim for this project.

# 1. Exploration of Dataset

## 1.1 Dataset Overview

### Step 1: Import the library and read the data



In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'raw').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'googleplaystore.csv'

import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")

In [ ]:
pd.read_csv(DATA_PATH)

### Step 2: Review the previous few pieces of data



In [ ]:
df.head()

In [ ]:
print(df.head())

### Step 3: Check the size of the dataset

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

### Step 4: View all variable names

In [ ]:
print(df.columns.tolist())

### Step 5: Check the data type

In [ ]:
df.info()

### Step 6: Check for missing values

In [ ]:
missing_values = df.isnull().sum()

print(missing_values)

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

print(missing_summary)

### Step 7: Check for duplicate data

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicated rows:", duplicate_count)

### Step 8: View the basic description statistics

In [ ]:
df.describe()

### Step 9: View categorical variables

In [ ]:
df.describe(include="object")

### Step 10: Check the number of unique values for each variable

In [ ]:
unique_summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Unique Values": df.nunique(),
    "Missing Values": df.isnull().sum()
})

print(unique_summary)

### Step 11: Classify the variables first

In [ ]:
categorical_features = [
    "Category",
    "Type",
    "Content Rating",
    "Genres"
]

potential_numeric_features = [
    "Rating",
    "Reviews",
    "Size",
    "Installs",
    "Price"
]

temporal_features = [
    "Last Updated"
]

identifier_or_technical_features = [
    "App",
    "Current Ver",
    "Android Ver"
]

print("Categorical features:")
print(categorical_features)

print("\nPotential numerical features:")
print(potential_numeric_features)

print("\nTemporal features:")
print(temporal_features)

print("\nIdentifier / technical features:")
print(identifier_or_technical_features)

### Dataset Overview

### Initial Dataset Observations

- The dataset contains 10,841 records and 13 original attributes.
- It contains categorical, quantitative, temporal and technical information about Google Play Store applications.
- Several conceptually numerical variables such as Reviews, Size, Installs and Price are stored as object/string data.
- Rating contains missing values.
- Further data conditioning is therefore required before exploratory visualisation and AI modelling.

## 1.2 Data Quality Diagnosis

##### Step 1: Establish an overall quality checklist

In [ ]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

quality_report

In [ ]:
#####print(quality_report)####in pycharm

#### Step2 ：Missing Values

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

In [ ]:
missing_percent = (
    df.isnull().mean() * 100
).round(2)

missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": missing_percent
})

missing_report = missing_report[
    missing_report["Missing Count"] > 0
].sort_values("Missing %", ascending=False)

missing_report

#### Step 3: Why is Rating particularly important?

### Observation

Rating contains a substantial number of missing values.
This issue is particularly important if Rating is subsequently selected
as the target variable for AI modelling.

#### Step 4: Check for duplicate records

In [ ]:
duplicate_rows = df.duplicated().sum()

print("Duplicated rows:", duplicate_rows)

In [ ]:
duplicate_apps = df["App"].duplicated().sum()

print("Duplicated App names:", duplicate_apps)

In [ ]:
df[df["App"].duplicated(keep=False)].sort_values("App").head(20)

In [ ]:
####df.drop_duplicates()

#### Step5 :check Reviews

In [ ]:
print(df["Reviews"].dtype)

In [ ]:
print(df["Reviews"].head(20))

In [ ]:
print(df["Reviews"].unique()[:30])

In [ ]:
reviews_numeric = pd.to_numeric(
    df["Reviews"],
    errors="coerce"
)

print("Values that cannot be converted:")
print(df.loc[reviews_numeric.isna(), "Reviews"].unique())

### Step 6: Check Installs

This is a very typical data cleaning problem.



In [ ]:
print(df["Installs"].head(20))

In [ ]:
df["Installs"].dtype

In [ ]:
print(df["Installs"].value_counts().sort_index())

In [ ]:
print(df["Installs"].unique())

#### Step 7: Check the Price

In [ ]:
print(df["Price"].unique()[:50])

In [ ]:
print(df["Price"].value_counts().head(20))

Price is stored as a string because of the currency symbol.
It must be converted into a numerical variable before quantitative
analysis or modelling.

#### Step 8: Check the Size

In [ ]:
print(df["Size"].unique()[:50])

In [ ]:
print(
    df["Size"]
    .astype(str)
    .str.contains("Varies with device", case=False)
    .sum()
)

#### Step 9: Check the Type

In [ ]:
print(df["Type"].value_counts(dropna=False))

#### Step 10: Check the Category

In [ ]:
print("Number of categories:", df["Category"].nunique())

print(
    df["Category"]
    .value_counts()
    .sort_values(ascending=False)
)

#### Step 11: Check the Content Rating

In [ ]:
print(
    df["Content Rating"]
    .value_counts(dropna=False)
)

#### Step 12: Check Last Updated

In [ ]:
print(df["Last Updated"].head())

In [ ]:
last_updated_datetime = pd.to_datetime(
    df["Last Updated"],
    errors="coerce"
)

print("Invalid dates:", last_updated_datetime.isna().sum())

#### Step 13: Check for outliers/illegal records

In [ ]:
for column in df.columns:
    print("\n", "=" * 50)
    print("COLUMN:", column)
    print("=" * 50)

    print(df[column].value_counts(dropna=False).head(10))

In [ ]:
print(
    df["Rating"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
df[
    (df["Rating"] < 0) |
    (df["Rating"] > 5)
]

In [ ]:
# Initialise the working copy before its first use.
df_clean = df.copy()

In [ ]:
invalid_rows = df_clean[
    (df_clean["Rating"] < 0) |
    (df_clean["Rating"] > 5)
]

invalid_rows

In [ ]:
df_clean = df_clean[
    df_clean["Rating"].isna() |
    df_clean["Rating"].between(0, 5)
].copy()

In [ ]:
df_clean["Rating"].describe()

In [ ]:
df_clean["Type"].value_counts(dropna=False)
df_clean["Category"].value_counts()

### Data Quality Diagnosis Summary

## 1.3 Data Conditioning and Cleaning

#### Step 1: malformed records first

In [ ]:
invalid_rating = df_clean[
    (df_clean["Rating"] < 0) |
    (df_clean["Rating"] > 5)
]

invalid_rating

##### 1.3.1 Invalid Record Inspection

Rating values outside the valid range were inspected before further
data conditioning to identify potentially malformed records.

#### Step 2: Handle completely duplicate records

In [ ]:
print("Before:", df_clean.shape)

print("Exact duplicate rows:",
      df_clean.duplicated().sum())

In [ ]:
df_clean = df_clean.drop_duplicates().copy()

print("After:", df_clean.shape)

#### Step 3: Clean the Reviews

In [ ]:
df_clean["Reviews"] = pd.to_numeric(
    df_clean["Reviews"],
    errors="coerce"
)

In [ ]:
print(df_clean["Reviews"].dtype)

print(
    "Missing Reviews after conversion:",
    df_clean["Reviews"].isnull().sum()
)

#### Step 4: Clean Installs

In [ ]:
df_clean["Installs"] = (
    df_clean["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

In [ ]:
df_clean["Installs"] = pd.to_numeric(
    df_clean["Installs"],
    errors="coerce"
)

In [ ]:
print(df_clean["Installs"].dtype)

print(
    "Missing Installs after conversion:",
    df_clean["Installs"].isnull().sum()
)

#### Step 5 clean price 

In [ ]:
df_clean["Price"] = (
    df_clean["Price"]
    .astype(str)
    .str.replace("$", "", regex=False)
)

In [ ]:
df_clean["Price"] = pd.to_numeric(
    df_clean["Price"],
    errors="coerce"
)

In [ ]:
print(df_clean["Price"].dtype)

print(
    "Missing Price after conversion:",
    df_clean["Price"].isnull().sum()
)

#### Step6 Size

In [ ]:
def convert_size_to_mb(value):
    value = str(value).strip()

    if value == "Varies with device":
        return np.nan

    if value.endswith("M"):
        return float(value[:-1])

    if value.endswith("k"):
        return float(value[:-1]) / 1024

    return np.nan

In [ ]:
df_clean["Size_MB"] = df_clean["Size"].apply(
    convert_size_to_mb
)

In [ ]:
print(df_clean[["Size", "Size_MB"]].head(20))

print(
    "Missing Size_MB:",
    df_clean["Size_MB"].isnull().sum()
)

#### Step 7 clean last updated

In [ ]:
df_clean["Last Updated"] = pd.to_datetime(
    df_clean["Last Updated"],
    errors="coerce"
)

In [ ]:
print(df_clean["Last Updated"].dtype)

print(
    "Invalid dates:",
    df_clean["Last Updated"].isnull().sum()
)

In [ ]:
df_clean["Updated_Year"] = (
    df_clean["Last Updated"].dt.year
)

#### Step 8 Rating Missing Values

In [ ]:
print(
    "Missing Rating:",
    df_clean["Rating"].isnull().sum()
)

In [ ]:
df_clean["Rating"].dropna()

In [ ]:
df_model = df_clean.dropna(
    subset=["Rating"]
).copy()

#### Step 9: Handle a few missing items such as Type/Content Rating

In [ ]:
df_clean[
    df_clean["Type"].isnull()
]

In [ ]:
df_clean[
    df_clean["Content Rating"].isnull()
]

#### Step 11: Verification after cleaning is completed

In [ ]:
df_clean.info()

In [ ]:
df_clean.isnull().sum()

In [ ]:
df_clean.describe()

#### Step12 Before vs After

In [ ]:
print("RAW DATA")
print(df.shape)

print("\nCLEANED DATA")
print(df_clean.shape)

In [ ]:
comparison = pd.DataFrame({
    "Raw Type": df.dtypes.astype(str),
    "Cleaned Type": df_clean[
        df.columns.intersection(df_clean.columns)
    ].dtypes.astype(str)
})

comparison

## 1.4 Exploratory Visualisation

### 1.4.1 Rating Distribution

The first question

How is the Rating of Google Play apps distributed as a whole？

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df_clean,
    x="Rating",
    bins=20,
    kde=True
)

plt.title("Distribution of Google Play Store App Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Apps")

plt.show()

###    这张图为什么重要？因为 Rating 很可能后面成为 AI target。
    你现在首先应该知道它：
    
    集中在哪里
    是否偏态
    有没有极端值
    variance 大不大
    (最后把这块删掉！！！！！！！           )

In [ ]:
df_clean["Rating"].describe()

### Observation

The rating distribution is concentrated toward the higher end of the
rating scale, indicating that app ratings are not evenly distributed.
This characteristic should be considered in subsequent modelling and
evaluation.

### 1.4.2 Category Distribution

The second question

Which categories do the apps in the dataset mainly come from?


In [ ]:
category_counts = (
    df_clean["Category"]
    .value_counts()
    .sort_values(ascending=False)
)

category_counts.head(10)

In [ ]:
top10_categories = category_counts.head(10)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=top10_categories.values,
    y=top10_categories.index
)

plt.title("Top 10 App Categories by Number of Apps")
plt.xlabel("Number of Apps")
plt.ylabel("Category")

plt.show()

#### 为什么只 Top 10？

因为老师评分里面有：

Appropriateness

和：

Human Visual Perception Principles

各占 25%。

如果 30 多个 Category 全部塞进去：

小字
密密麻麻
颜色很多

反而不利于 perception。

所以：

排序 + Top 10 + horizontal bar chart

是更合适的设计。

### 这里重点观察什么？

例如：

哪些 Category 样本最多？
数据是否不平衡？
某些类别是否非常少？

这会影响之后：

Category vs Rating

和 AI modelling。

(！！！！！这个也是最后删掉！！！！！！)

#### 1.4.3 Free vs Paid Apps

The third question

What is the difference in the number of Free and Paid apps?

In [ ]:
type_counts = df_clean["Type"].value_counts()

type_counts

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(
    x=type_counts.index,
    y=type_counts.values
)

plt.title("Distribution of Free and Paid Apps")
plt.xlabel("App Type")
plt.ylabel("Number of Apps")

plt.show()

###这一张可能发现

如果 Free 数量远大于 Paid：

数据存在明显的 category imbalance。

这个 observation 后面非常有用。

因为如果你比较：

Paid apps rating 是否更高？

必须记住：

两组 sample size 差很多。

（！！！！ 这个最后删掉！！！！）

### 1.4.4 Installs Distribution

In [ ]:
df_clean["Installs"].describe()

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(
    data=df_clean,
    x="Installs",
    bins=30
)

plt.xscale("log")

plt.title("Distribution of App Install Counts")
plt.xlabel("Installs (Log Scale)")
plt.ylabel("Number of Apps")

plt.show()

为什么这里用 log scale？

不是为了把数据改掉。

只是为了：

让高度右偏、跨度很大的 distribution 在图上可读。

原始数据依然是：

df_clean["Installs"]

没有改变。

这是一个非常典型的：

Visualization design decision

符合 CA6002 重点。

（！！！ 最后删掉！！！）

#### 1.4.5 Reviews Distribution

In [ ]:
df_clean["Reviews"].describe()

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(
    data=df_clean,
    x="Reviews",
    bins=30
)

plt.xscale("log")

plt.title("Distribution of App Review Counts")
plt.xlabel("Number of Reviews (Log Scale)")
plt.ylabel("Number of Apps")

plt.show()

这里你很可能会发现：

大量 App 的 Reviews 很少，而少数 App 拥有极高 Reviews。

这对后面很重要。

因为：

Rating = 5.0
Reviews = 2

和：

Rating = 4.8
Reviews = 2,000,000

它们的可信度显然不是一回事。

这就可能自然引出后面的 Visual Storytelling。

（！！！！ 最后删掉！！！！）

#### 1.4.6 Rating by App Type

到这里开始第一个简单 relationship。

Q: Are the Rating distributions of the Free and Paid apps different?

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_clean,
    x="Type",
    y="Rating"
)

plt.title("Rating Distribution by App Type")
plt.xlabel("App Type")
plt.ylabel("Rating")

plt.show()

Boxplot 可以同时看到：

median
IQR
spread
outliers

信息丰富得多。

（！！！！ 最后删掉！！！！）

#### 1.4.7 Rating by Major Categories

但千万不要全部 Category 一起 boxplot。

先选样本量最多的 Top 10：

（！！！！ 最后删掉！！！！）

In [ ]:
top_categories = (
    df_clean["Category"]
    .value_counts()
    .head(10)
    .index
)

category_rating = df_clean[
    df_clean["Category"].isin(top_categories)
]

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=category_rating,
    x="Category",
    y="Rating"
)

plt.xticks(rotation=45, ha="right")

plt.title("Rating Distribution Across Major App Categories")
plt.xlabel("Category")
plt.ylabel("Rating")

plt.tight_layout()
plt.show()

这张图回答：

不同 Category 的 Rating distribution 是否有明显差异？

不要只看哪一组平均值最高。

你要观察：

median
spread
outliers
category-to-category variation

（！！！！ 最后删掉！！！！）

## 1.5 Relationship and Feature Investigation

This section investigates relationships between app characteristics and
Rating to identify potentially informative features for subsequent AI
modelling.

#### 1.5.1 Reviews vs Rating

#### The first question
Is there any relationship between the number of Reviews and the Rating?

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df_clean,
    x="Reviews",
    y="Rating",
    alpha=0.3
)

plt.xscale("log")

plt.title("Relationship Between Reviews and App Rating")
plt.xlabel("Number of Reviews (Log Scale)")
plt.ylabel("Rating")

plt.show()

注意 Reviews = 0

log scale 无法表示 0。

In [ ]:
reviews_rating = df_clean[
    (df_clean["Reviews"] > 0) &
    (df_clean["Rating"].notna())
]

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=reviews_rating,
    x="Reviews",
    y="Rating",
    alpha=0.3
)

plt.xscale("log")

plt.title("Relationship Between Reviews and App Rating")
plt.xlabel("Number of Reviews (Log Scale)")
plt.ylabel("Rating")

plt.show()

这里观察：

Review 数量增加时 Rating 有没有明显变化？

高 Review App 的 Rating 是否更集中？

Review 很少的 App 是否 Rating 分散更大？


#### 1.5.2 加 LOWESS 看趋势

In [ ]:
df_clean["Log_Reviews"] = np.log1p(
    df_clean["Reviews"]
)

In [ ]:
plt.figure(figsize=(9, 6))

sns.regplot(
    data=df_clean,
    x="Log_Reviews",
    y="Rating",
    lowess=True,
    scatter_kws={"alpha": 0.2},
    line_kws={"linewidth": 2}
)

plt.title("LOWESS Trend: Reviews vs App Rating")
plt.xlabel("log(1 + Reviews)")
plt.ylabel("Rating")

plt.show()

散点：

每一个 App。

LOWESS 曲线：

Reviews 与 Rating 的局部总体趋势。

不是在训练 AI。

也不是：

Linear Regression。

LOWESS 在这里是 exploratory visualisation technique。

1.5.3 Installs vs Rating

第二个重要变量：

安装量越高，Rating 会不会越高？

In [ ]:
installs_rating = df_clean[
    (df_clean["Installs"] > 0) &
    (df_clean["Rating"].notna())
]

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=installs_rating,
    x="Installs",
    y="Rating",
    alpha=0.3
)

plt.xscale("log")

plt.title("Relationship Between Installs and App Rating")
plt.xlabel("Number of Installs (Log Scale)")
plt.ylabel("Rating")

plt.show()

In [ ]:
df_clean["Log_Installs"] = np.log1p(
    df_clean["Installs"]
)

In [ ]:
plt.figure(figsize=(9, 6))

sns.regplot(
    data=df_clean,
    x="Log_Installs",
    y="Rating",
    lowess=True,
    scatter_kws={"alpha": 0.2},
    line_kws={"linewidth": 2}
)

plt.title("LOWESS Trend: Installs vs App Rating")
plt.xlabel("log(1 + Installs)")
plt.ylabel("Rating")

plt.show()

现在比较：

Reviews → Rating

和：

Installs → Rating

哪个关系看起来更明显？

##### 1.5.4 Size vs Rating

接下来调查：

App 大小是否和 Rating 有关系？

In [ ]:
size_rating = df_clean[
    df_clean["Size_MB"].notna() &
    df_clean["Rating"].notna()
]

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=size_rating,
    x="Size_MB",
    y="Rating",
    alpha=0.3
)

plt.title("Relationship Between App Size and Rating")
plt.xlabel("App Size (MB)")
plt.ylabel("Rating")

plt.show()

先看原始关系。

如果图显示：

Size 从小到大，Rating 基本没有明显趋势

那也是一个有效发现。

“没有明显 relationship”本身也是结果。

#### 1.5.5 Price vs Rating

#### A. Free vs Paid

In [ ]:
sns.boxplot(
    data=df_clean,
    x="Type",
    y="Rating"
)

#### B. Paid Apps 内部

In [ ]:
paid_apps = df_clean[
    (df_clean["Price"] > 0) &
    (df_clean["Rating"].notna())
]

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=paid_apps,
    x="Price",
    y="Rating",
    alpha=0.4
)

plt.title("Relationship Between Price and Rating for Paid Apps")
plt.xlabel("Price")
plt.ylabel("Rating")

plt.show()

这里回答：

在 Paid Apps 内部，更贵是不是意味着 Rating 更高？

#### 1.5.6 Last Updated / Updated Year vs Rating

In [ ]:
df_clean.groupby("Updated_Year")["Rating"].agg(
    ["count", "mean", "median"]
)

In [ ]:
year_rating = (
    df_clean
    .groupby("Updated_Year")["Rating"]
    .agg(["mean", "count"])
    .reset_index()
)

plt.figure(figsize=(9, 5))

sns.lineplot(
    data=year_rating,
    x="Updated_Year",
    y="mean",
    marker="o"
)

plt.title("Average App Rating by Last Updated Year")
plt.xlabel("Last Updated Year")
plt.ylabel("Average Rating")

plt.show()

#### 1.5.7 Numerical Correlation Analysis

In [ ]:
numeric_features = [
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Size_MB"
]

correlation_matrix = (
    df_clean[numeric_features]
    .corr()
)

correlation_matrix

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    square=True
)

plt.title("Correlation Matrix of Numerical App Features")

plt.tight_layout()
plt.show()

#### 1.5.8 Doing Spearman Correlation again

In [ ]:
spearman_corr = (
    df_clean[numeric_features]
    .corr(method="spearman")
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    spearman_corr,
    annot=True,
    fmt=".2f",
    square=True
)

plt.title("Spearman Correlation of Numerical App Features")

plt.tight_layout()
plt.show()

Pearson
vs
Spearman

比较：

Pearson
vs
Spearman

为什么？

Pearson

更关注：

linear relationship

Spearman

更关注：

monotonic relationship / rank association

对于高度偏斜的数据，Spearman 可以提供一个补充视角。

不过不要看到 Spearman 比 Pearson 高就说：

Spearman 更正确。

应该说：

两者衡量的 relationship 不完全一样。

#### 1.5.9 Category vs Rating

In [ ]:
category_stats = (
    df_clean
    .groupby("Category")["Rating"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

category_stats

#### 1.5.10 Content Rating vs Rating

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_clean,
    x="Content Rating",
    y="Rating"
)

plt.xticks(rotation=30, ha="right")

plt.title("Rating Distribution by Content Rating")
plt.xlabel("Content Rating")
plt.ylabel("Rating")

plt.tight_layout()
plt.show()

回答：

不同 target audience 的 App rating distributions 是否不同？

### Candidate Feature Investigation Summary

| Candidate Feature | Investigation | Relationship with Rating | Decision |
|---|---|---|---|
| Reviews | Scatter + LOWESS + Pearson/Spearman | Weak positive association; LOWESS indicates a nonlinear pattern and ratings become more concentrated as review volume increases | Keep |
| Installs | Scatter + LOWESS + Pearson/Spearman | Very weak overall association, with a slight nonlinear pattern visible in LOWESS | Keep |
| Size_MB | Scatter + Pearson/Spearman | Very weak positive association; no strong direct relationship | Keep / Investigate |
| Price | Paid-app scatter + Pearson/Spearman | Little direct association with Rating; extreme high-price observations are present | Keep / Investigate |
| Type | Boxplot | Free and paid apps show broadly similar rating distributions | Keep |
| Category | Boxplot + Group Statistics | Rating distributions vary across categories, although category sample sizes are highly unequal | Keep |
| Content Rating | Boxplot | Some variation exists between audience groups, but distributions substantially overlap | Keep / Investigate |
| Updated Year | Temporal comparison | More recently updated apps show somewhat higher average ratings, but yearly sample sizes are highly imbalanced | Keep |

### Key Finding

No individual numerical feature shows a strong direct relationship with
Rating. Pearson correlations with Rating are weak for Reviews, Installs,
Price and Size_MB.

The LOWESS and Spearman analyses nevertheless suggest that some
relationships may be nonlinear or weakly monotonic. Therefore, these
features should not be excluded solely based on Pearson correlation.

This motivates the use of a multivariate AI model that can investigate
the combined and potentially nonlinear effects of multiple app
characteristics on Rating.

## 1.6 Feature Selection and Final Modelling Dataset

Based on the exploratory analysis and relationship investigation,
candidate features are selected for subsequent AI modelling.

### Step 1: First, formally determine the Target



In [ ]:
target = "Rating"

print("Target variable:", target)

### Target Variable

**Rating** is selected as the target variable.

The subsequent AI task will investigate whether an app's rating can be
predicted from its app characteristics, market adoption, user engagement,
pricing, audience and update information.

#### Step 2: Determine the Candidate Features

In [ ]:
selected_features = [
    "Reviews",
    "Installs",
    "Size_MB",
    "Price",
    "Type",
    "Category",
    "Content Rating",
    "Updated_Year"
]

selected_features

##### Step 3: Clarify which variables are not to be used

#### Step 4： Feature Selection Summary

### Feature Selection Summary

| Feature | Decision | Rationale |
|---|---|---|
| Reviews | Keep | Represents user engagement and showed weak but potentially nonlinear association with Rating |
| Installs | Keep | Represents market adoption and may contain nonlinear information |
| Size_MB | Keep | Standardised numerical representation of app size |
| Price | Keep | Represents pricing strategy |
| Type | Keep | Distinguishes free and paid apps |
| Category | Keep | Captures differences among app categories |
| Content Rating | Keep | Represents target audience |
| Updated_Year | Keep | Captures app update recency |
| App | Exclude | Primarily an identifier with very high cardinality |
| Size | Exclude | Replaced by the standardised Size_MB feature |
| Genres | Exclude | Overlaps with Category and has more complex categorical structure |
| Last Updated | Exclude | Replaced by Updated_Year |
| Current Ver | Exclude | Irregular high-cardinality version information |
| Android Ver | Exclude | Irregular version information and not central to the current objective |

#### Step 5：Final Modelling Dataset

In [ ]:
model_columns = selected_features + [target]

df_model = df_clean[model_columns].copy()

df_model.head()

In [ ]:
df

In [ ]:
df_clean

In [ ]:
df_model

#### Step 6: Handle Target Missing

In [ ]:
print(
    "Rows before removing missing target:",
    df_model.shape[0]
)

df_model = df_model.dropna(
    subset=["Rating"]
).copy()

print(
    "Rows after removing missing target:",
    df_model.shape[0]
)

#### Step 7: Check for Missing features

In [ ]:
df_model.isnull().sum()

In [ ]:
feature_missing = df_model.isnull().sum()

feature_missing[
    feature_missing > 0
]

#### Step 8: Separate Numerical and Categorical Features

In [ ]:
numerical_features = [
    "Reviews",
    "Installs",
    "Size_MB",
    "Price",
    "Updated_Year"
]

categorical_features = [
    "Type",
    "Category",
    "Content Rating"
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

#### Step 9: Final Inspection

In [ ]:
print("Final modelling dataset shape:")
print(df_model.shape)

print("\nSelected features:")
print(selected_features)

print("\nTarget:")
print(target)

print("\nData types:")
print(df_model.dtypes)

print("\nMissing values:")
print(df_model.isnull().sum())

In [ ]:
df_model.head()

#### Step 11: Finally, write the Exploration summary

### Exploration of Dataset Summary

The exploration and conditioning process transformed the raw Google Play
Store dataset into a modelling dataset suitable for subsequent AI
development.

Rating was selected as the prediction target. Eight candidate features
were retained to represent user engagement, market adoption, app
characteristics, pricing, category, audience and update recency.

Exploratory visualisation and correlation analysis showed that no single
numerical feature has a strong direct relationship with Rating. However,
LOWESS trends, categorical comparisons and rank-based associations suggest
that nonlinear and combined relationships may exist.

These findings motivate the subsequent investigation of multivariate
regression models capable of learning relationships between multiple app
characteristics and Rating.